# 07 — Deduplication

Everything you did in **02–06** (exact rules, measurement loop, fuzzy, design, blocking) applies here—but on a **single table**. Find rows that represent the same entity (duplicates) even when IDs differ. `Deduplicator` wraps the same ideas: same `match(rules=...)`, `match_fuzzy(...)`, `blocking_key`, and `evaluate()` so you don't learn a new API. We use the sample deduplication set (1000 rows, 50 known duplicate pairs).

**Back:** [06 Blocking](06_blocking.ipynb) · **TOC:** [Preamble & TOC](00_preamble_and_toc.ipynb)

## 1. Load data and ground truth

For deduplication, ground truth is "pairs of row IDs that are the same entity." You might get that from a golden set or from a rule (e.g. same email ⇒ duplicate). **Data:** This set has **1000 rows**: 50 duplicate pairs (same email, two rows each) and 900 unique rows. The loader returns the table and a ground-truth DataFrame of those 50 pairs.

In [1]:
import sys
from pathlib import Path
import polars as pl
from matcher import Deduplicator

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_deduplication

df, ground_truth = load_deduplication()
print(f"Source: {df.shape[0]} rows")
print(f"Ground truth: {ground_truth.shape[0]} duplicate pairs")
print(ground_truth.head(5))

Source: 1000 rows
Ground truth: 50 duplicate pairs
shape: (5, 2)
┌─────────┬──────────┐
│ left_id ┆ right_id │
│ ---     ┆ ---      │
│ str     ┆ str      │
╞═════════╪══════════╡
│ dup_1   ┆ dup_2    │
│ dup_3   ┆ dup_4    │
│ dup_5   ┆ dup_6    │
│ dup_7   ┆ dup_8    │
│ dup_9   ┆ dup_10   │
└─────────┴──────────┘


## 2. Exact matching with Deduplicator

`Deduplicator(source=df, id_col="id")` treats your table as both "left" and "right" under the hood (as in 02). `match(rules=...)` returns duplicate pairs; self-pairs (same id on both sides) are dropped automatically. Same rules and blocking options as in 02–06.

In [2]:
deduplicator = Deduplicator(source=df, id_col="id")
results = deduplicator.match(rules="email")
print(f"Duplicate pairs (email rule): {results.count}")
print(results.matches.select(["id", "id_right", "email"]).head(5))

Duplicate pairs (email rule): 100
shape: (5, 3)
┌─────────┬──────────┬──────────────────────────┐
│ id      ┆ id_right ┆ email                    │
│ ---     ┆ ---      ┆ ---                      │
│ str     ┆ str      ┆ str                      │
╞═════════╪══════════╪══════════════════════════╡
│ dup_45  ┆ dup_46   ┆ duplicate_22@example.com │
│ dup_100 ┆ dup_99   ┆ duplicate_49@example.com │
│ dup_18  ┆ dup_17   ┆ duplicate_8@example.com  │
│ dup_98  ┆ dup_97   ┆ duplicate_48@example.com │
│ dup_70  ┆ dup_69   ┆ duplicate_34@example.com │
└─────────┴──────────┴──────────────────────────┘


## 3. Evaluate

Same `evaluate()` API as in 02–06. The match table has `id` and `id_right`, so we pass `left_id_col="id"` and `right_id_col="id_right"`. Then you're in the same loop: change rules or blocking, re-run, compare metrics (precision, recall, F1).

In [3]:
metrics = results.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Precision: {metrics['precision']:.2%}, Recall: {metrics['recall']:.2%}, F1: {metrics['f1']:.2%}")
print(f"True positives: {metrics['true_positives']}, False positives: {metrics['false_positives']}, False negatives: {metrics['false_negatives']}")

Precision: 50.00%, Recall: 100.00%, F1: 66.67%
True positives: 50, False positives: 50, False negatives: 0


## 4. Optional: fuzzy and blocking

Fuzzy and blocking work the same as in 04 and 06: `match_fuzzy(field=..., threshold=...)` and `blocking_key=...` on both `match()` and `match_fuzzy()`. Self-matches are still excluded. So the full path from 02 (exact) through 06 (blocking) transfers—one table instead of two.

In [4]:
results_fuzzy = deduplicator.match_fuzzy(field="first_name", threshold=0.8)
print(f"Fuzzy duplicate pairs (first_name, 0.8): {results_fuzzy.count}")
results_block = deduplicator.match(rules="email", blocking_key="zip_code")
print(f"Exact with blocking (zip_code): {results_block.count}")

Fuzzy duplicate pairs (first_name, 0.8): 819000
Exact with blocking (zip_code): 100


---

**End of tutorial.** You've walked the full path: [01](01_preparation.ipynb) preparation → [02](02_exact_matching.ipynb) exact → [03](03_measurement_loop.ipynb) measurement loop → [04](04_fuzzy_matching.ipynb) fuzzy → [05](05_design_algorithm.ipynb) design (combos, blended) → [06](06_blocking.ipynb) blocking → [07](07_deduplication.ipynb) deduplication. The habit that ties it together: **measure, tune, compare.** For more, see the [main README](../../README.md) and the [design docs](../MATCHING_ALGORITHM_DESIGN.md) in `docs/`.